# afMLevel Demonstration Notebook

This notebook provides an introduction to the **afMLevel** package and demonstrates how to use its trained U‑Net models to level Atomic Force Microscopy (AFM) images. Two machine‑learning–based levelling approaches are demonstrated:

1. **Background Model**
   This model predicts the background artifacts present in an AFM image, such as tilt, line shift and other common imaging artifacts. The predicted
   background is then subtracted from the original image to produce a levelled AFM image. This workflow is implemented in the `level_ml_bg()` function.
   
2. **Mask Model**
   The mask model uses a trained U-Net to generate a feature mask for an AFM image. This mask is then used in a conventional levelling pipeline involving
   plane and line fits, so that the background (unmasked areas) drives the levelling procedure. The mask generation and levelling operations are
   orchestrated by the `level_ml_mask()` function.

To explore how the models work, simply work through this notebook by clicking on each code cell and running it (using **Shift** + **Enter**) to view the output. You can experiment with different parameters by editing the code, and re‑running a cell will immediately show how your changes affect the results. You can also switch from the provided demonstration data to your own AFM datasets by adjusting the file paths. Notebooks are designed for exploration, so you can rerun cells or restart the kernel at any time without worrying about breaking anything.

## 1. Setup and Imports

Start by ensuring all the required packages and functions are installed and imported. The two trained U-Net models are downloaded if not already available and the  `lutAFM` AFM image colourmap loaded.

If you do not have **afMLevel** already installed or want to process your own data from common AFM file formats (other than tiff files), uncomment the relevant lines in the next block to install the required packages. 

In [ ]:

#Ignore this cell if you have installed afMLevel with the `pip install -e .[notebooks]` command, all the notebook dependancies will already be installed.

# If afMLevel isn't installed ensure this notebook is opened from the
# repository root and uncomment the installation line (starting !).
# Install in editable mode if developing:
# !pip install -e .

# If you want to load your own data from common AFM file formats uncomment
# the following line and run the installation line to install the AFMReader
# package that can open .jpk, .h5-jpk, .spm, .ibw etc. 
# !pip install AFMReader


Now import the required packages, download the trained models if required and load the AFM colourmap.

In [ ]:

from pathlib import Path
import os
import requests
from tifffile import imread
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

from afmlevel.background_model import level_ml_bg
from afmlevel.mask_model import level_ml_mask, ml_mask, ml_edges

# Load AFM colourmap
AFM = np.load('./lutAFM.npy') # by default the working directory is the folder the notebook is in, i.e. /notebooks
AFM = ListedColormap(AFM)


## 2. Loading Data

Next step is to load the AFM image that we will apply the **afMLevel** levelling routines to. Example data can be downloaded for the demonstration, alternatively by running the following cell, this can be skipped, and you can run the demo on your own AFM image by commenting and uncommenting (with #) the indicated lines below and providing a path to your own data.

The example data is from Meinhardt, A., Qi, P., David, C., Maximov, I., & Keller, T. F. (2024). Raw data for manuscript A. Meinhardt et al., A pathway towards sub-10 nm surface nanostructures utilizing block copolymer crystallization control. https://doi.org/10.5281/zenodo.14202742

We can then visualise the raw data with Matplotlib. 

In [ ]:

# Download example data from this dataset: https://zenodo.org/records/14202742

import zipfile

# Define URLs and paths
zip_url = "https://zenodo.org/records/14202742/files/Figure_6-R.zip?download=1"
local_zip = Path("data/dataset_nano.zip")
extract_dir = Path("data/extracted_nano")

extract_dir.mkdir(parents=True, exist_ok=True)

# Download the zip file
if not local_zip.exists():
    print("Downloading dataset...")
    r = requests.get(zip_url, stream=True)
    r.raise_for_status()

    with open(local_zip, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
else:
    print("Using cached download:", local_zip)

# Unzip
print("Extracting...")
with zipfile.ZipFile(local_zip, "r") as z:
    z.extractall(extract_dir)

print("Done. Files extracted to:", extract_dir)


In [ ]:

def load_with_afmreader(path : str, channel: str):
    """Use the general loader from AFMReader to open AFM file formats."""
    from AFMReader.general_loader import LoadFile
    loader = LoadFile(path, channel)
    data, _ = loader.load()
    return data


# This path points to where the sample data is downloaded to (comment this out with a # if using your own data).
data_path = Path("data/extracted_nano/Figure_6-R/230602_M3M_PE-b-PEO_annealed.0_00006.spm")
# The SPM sample data has 'Height Sensor', 'Amplitude Error', 'Phase' channels. Choose one that contains topography data i.e. "Height Sensor"
channel = "Height Sensor"

# To use your own data uncomment and update the following line(s):
# data_path = Path("/path/to/your/data.tiff")
# Data can be tiff files or raw SPM, JPK, GWY, IBW, STP, H5-JPK, ASD
# channel = "height_retrace" # change this to the AFM data channel you want to read
frame = 0 # If loading 'video' formats i.e. .asd or .h5-jpk set this to the frame you want to work on.

#Now determine the data type and try to load the data
_, ext = os.path.splitext(data_path)
if ext.lower() in [".tif", ".tiff"]:
    data = imread(data_path)
elif ext.lower() in [".asd", ".h5-jpk"]:
    afm_stack = load_with_afmreader(data_path, channel)
    data = afm_stack[frame] 
elif ext[1:].isdigit() and len(ext) == 4:   # patten for old Bruker files
    from AFMReader.spm import load_spm
    data, _ = load_spm(data_path, channel)
else:
    data = load_with_afmreader(data_path, channel)

vmin, vmax = np.percentile(data, [1, 99])
plt.figure()
plt.title("Raw AFM Data")
plt.imshow(data, cmap=AFM, vmin=vmin, vmax=vmax)
plt.colorbar(label="Height (nm)")
plt.show()


## 3. Apply Models

### Background Model

The background model is applied with the `level_ml_bg()` function within the `background_model` module. The function requires the loaded AFM data (NumPy array) and the path to the trained background model.

If the `background` parameter is set as `True` then rather than returning the levelled image, the background before it is subtracted is returned. 

In [ ]:

# Apply the background model
bg_levelled = level_ml_bg(data)

# Apply the background model and return the background
bg_background = level_ml_bg(data, background = True)

plt.figure(figsize=(15,4))

vmin, vmax = np.percentile(data, [1, 99])
plt.subplot(1,3,1)
plt.title("Raw Data")
plt.imshow(data, cmap=AFM, vmin=vmin, vmax=vmax)
plt.colorbar(label="Height (nm)")

vmin, vmax = np.percentile(bg_background, [1, 99])
plt.subplot(1,3,2)
plt.title("Background")
plt.imshow(bg_background, cmap=AFM, vmin=vmin, vmax=vmax)
plt.colorbar(label="Height (nm)")

vmin, vmax = np.percentile(bg_levelled, [1, 99])
plt.subplot(1,3,3)
plt.title("Levelled Data")
plt.imshow(bg_levelled, cmap=AFM, vmin=vmin, vmax=vmax)
plt.colorbar(label="Height (nm)")

plt.tight_layout()
plt.show()


### Mask Model

Masks are generated using the `ml_mask()` and `ml_edges()` functions, which return binary NumPy arrays of type `uint8` (values 0 or 1). The `level_ml_mask()` function coordinates levelling workflows; it applies plane and line‑fit corrections (from the `pnanolocz` library) and uses both afMLevel masking functions to mask features in the data prior to levelling steps. 

#### ml_mask

The `ml_mask()` function detects features within the image and produces a binary mask, enabling those regions to be omitted from the levelling operations.

#### ml_edges

The `ml_edges()` function builds on the output of `ml_mask()` to generate an edge‑specific mask. It first obtains a binary mask from `ml_mask()`, then applies a sequence of morphological operations, including perimeter extraction, removal of small objects, hole filling, and dilation, to isolate the edges of detected features. The resulting binary mask is compatible with the functions in the `level_weighted` module of `pnanolocz`.

In [ ]:

# Generate mask using trained U-Net model

mask = ml_mask(data)
edges = ml_edges(data)

plt.figure(figsize=(15,4))

vmin, vmax = np.percentile(data, [1, 99])
plt.subplot(1,3,1)
plt.title("Raw Data")
plt.imshow(data, cmap=AFM, vmin=vmin, vmax=vmax)

plt.subplot(1,3,2)
plt.title("Mask")
plt.imshow(mask, cmap="grey")

plt.subplot(1,3,3)
plt.title("Edges Mask")
plt.imshow(edges, cmap="grey")

plt.tight_layout()
plt.show()


#### level_ml_mask

These masking functions are used within `level_ml_mask()`. To level an AFM image using the mask model, a levelling routine must be selected from the available methods listed in the `DEFAULT_ML_ROUTINES` dictionary in the `mask_model` module.

In [ ]:

from afmlevel.mask_model import DEFAULT_ML_ROUTINES

routine_names = ", ".join(DEFAULT_ML_ROUTINES)
print(f"Available ML mask levelling routines: {routine_names}.")


In testing, the `iterative-ml-mask` routine provided the most consistent results across a wide range of images and is therefore set as the default option.

The levelling routines follow a sequence of levelling and masking steps similar to traditional manual or automated AFM image‑processing workflows. The key difference is that, instead of relying on classical thresholding or edge‑detection methods, the masks are generated by machine‑learning‑based functions (`ml_mask()` and `ml_edges()`), providing more consistent feature detection and more stable levelling performance when applied without manual parameter optimisation for each image.

In [ ]:

method = "iterative-ml-mask"
steps = DEFAULT_ML_ROUTINES[method]

print(f"Steps for {method}:")
for step in steps:
    name = step["func"].__name__
    if "method" in step:
        name += f" ({step['method']})"
    print(name)


The `level_ml_mask()` takes a NumPy array (`imarray`), the path to the trained mask U-Net model (`model_path`), and a `method` as inputs. 

In [ ]:

# Apply methods
mask_levelled = level_ml_mask(data, method="iterative-ml-mask")
mask_levelled_multi = level_ml_mask(data, method="multi-plane-ml-mask")

plt.figure(figsize=(15,4))

# --- Raw ---
vmin, vmax = np.percentile(data, [1, 99])
plt.subplot(1,3,1)
plt.title("Raw Data")
plt.imshow(data, cmap=AFM, vmin=vmin, vmax=vmax)
plt.colorbar(label="Height (nm)")

# --- iterative-ml-mask ---
vmin, vmax = np.percentile(mask_levelled, [1, 99])
plt.subplot(1,3,2)
plt.title('"iterative-ml-mask" Levelled')
plt.imshow(mask_levelled, cmap=AFM, vmin=vmin, vmax=vmax)
plt.colorbar(label="Height (nm)")

# --- Multi-plane-ml-mask ---
vmin, vmax = np.percentile(mask_levelled_multi, [1, 99])
plt.subplot(1,3,3)
plt.title('"multi-plane-ml-mask" Levelled')
plt.imshow(mask_levelled_multi, cmap=AFM, vmin=vmin, vmax=vmax)
plt.colorbar(label="Height (nm)")

plt.tight_layout()
plt.show()
